# Completed field pixels: footprints and crop separation

This notebook keeps **one sample per 10 m TESSERA pixel**. It never averages a field into one embedding. It can run while the pipeline is active: it snapshots only atomically published `w4` shards, verifies which fields have every expected pixel row, and then:

1. plots every available 10 × 10 m pixel footprint for example completed fields against the WKT;
2. plots those same pixels in a shared PCA embedding space; and
3. runs an exploratory crop-separation test on individual pixel embeddings with train/test folds grouped by `field_uid`.

Crop labels remain field-level labels inherited by their pixels. Shared pixel rows are excluded without discarding the field's private pixels. Identical geometries with one label retain one canonical field; conflicting-label geometry duplicates and every pixel they touch are excluded from testing.

In [ ]:
from collections import defaultdict
from pathlib import Path
import hashlib
import json
import os
import sys

from IPython import get_ipython
from IPython.display import display

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pyproj import Transformer

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "plain_tessera_incremental" / "__init__.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the SpectraJam checkout.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from plain_tessera_incremental.geometry import parse_field_geometry

OUTPUT_DIR = Path(
    os.environ.get(
        "TESSERA_OUTPUT_DIR",
        "/mnt/noobjam/harvard_tessera_incremental_v2",
    )
)
TARGET_WINDOW = os.environ.get("TESSERA_WINDOW_ID", "w4")
PIXEL_SIZE_M = 10
EMBEDDING_DIM = 128
MIN_PRIVATE_PIXELS_PER_FIELD = 1
MIN_FIELDS_PER_CROP = 5
MAX_CROPS = 8
TARGET_LABELS = None  # Or set a list such as ["Maize", "Wheat"].
MAX_FIELDS_PER_CROP = 48
MAX_PIXELS_PER_FIELD = 24
MAX_EXAMPLE_FIELD_PIXELS = 1000  # Select a full field no larger than this when possible.
N_EXAMPLE_FIELDS = 6
N_FOLDS = 5
N_PERMUTATIONS = 99
RANDOM_SEED = 20260708

def read_parquet_file(path, columns=None):
    # Read exactly one file; do not infer hive partitions from its directory.
    return pq.ParquetFile(path).read(columns=columns).to_pandas()

def parquet_metadata(path):
    raw = pq.ParquetFile(path).metadata.metadata or {}
    return {key.decode(): value.decode() for key, value in raw.items()}

def embedding_vector(value):
    if value is None:
        return None
    vector = np.asarray(value, dtype=np.float32)
    if vector.shape != (EMBEDDING_DIM,) or not np.isfinite(vector).all():
        return None
    return vector

def expected_row_id(run_fingerprint, field_uid, pixel_id, window_id):
    payload = f"{run_fingerprint}:{field_uid}:{pixel_id}:{window_id}"
    return hashlib.sha256(payload.encode()).hexdigest()

print("Output:", OUTPUT_DIR)
print("Window:", TARGET_WINDOW)

In [ ]:
required_paths = ["run.json", "fields.parquet", "pixels.parquet", "field_pixels.parquet"]
missing = [name for name in required_paths if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing pipeline artifacts: {missing}")

run = json.loads((OUTPUT_DIR / "run.json").read_text())
run_fingerprint = str(run["run_fingerprint"])
run_complete_at_snapshot = (OUTPUT_DIR / "COMPLETED.json").is_file()
manifest_windows = run.get("config", {}).get("windows", [])
if manifest_windows and TARGET_WINDOW not in {
    str(window.get("window_id")) for window in manifest_windows
}:
    raise RuntimeError(f"Window {TARGET_WINDOW!r} is not part of this run.")

for artifact_name in ("fields", "pixels", "field_pixels"):
    path = OUTPUT_DIR / f"{artifact_name}.parquet"
    metadata = parquet_metadata(path)
    expected = {
        "run_fingerprint": run_fingerprint,
        "artifact": artifact_name,
        "schema_version": "1",
    }
    mismatched = {
        key: (metadata.get(key), value)
        for key, value in expected.items()
        if metadata.get(key) != value
    }
    if mismatched:
        raise RuntimeError(f"Artifact metadata mismatch for {path}: {mismatched}")

shard_dir = OUTPUT_DIR / "embeddings" / f"window_id={TARGET_WINDOW}"
discovered_files = sorted(
    path
    for path in shard_dir.glob("*.parquet")
    if not path.name.endswith(".part")
)
if not discovered_files:
    raise RuntimeError(f"No published {TARGET_WINDOW} embedding shards are available.")

required_embedding_columns = {
    "row_id",
    "run_fingerprint",
    "field_uid",
    "pixel_id",
    "utm_epsg",
    "pixel_x_index",
    "pixel_y_index",
    "window_id",
    "pixel_longitude",
    "pixel_latitude",
    "landcover",
    "outcome",
    "embedding",
}
embedding_files = []
seen_task_keys = set()
for path in discovered_files:
    parquet = pq.ParquetFile(path)
    metadata = parquet_metadata(path)
    expected = {
        "run_fingerprint": run_fingerprint,
        "artifact": "pixel_embeddings",
        "schema_version": "1",
        "window_id": TARGET_WINDOW,
    }
    mismatched = {
        key: (metadata.get(key), value)
        for key, value in expected.items()
        if metadata.get(key) != value
    }
    if mismatched:
        raise RuntimeError(f"Shard metadata mismatch for {path}: {mismatched}")
    task_key = metadata.get("task_key")
    if not task_key or not metadata.get("task_fingerprint"):
        raise RuntimeError(f"Shard task metadata is missing: {path}")
    if task_key in seen_task_keys:
        raise RuntimeError(f"Duplicate {TARGET_WINDOW} task shard: {task_key}")
    seen_task_keys.add(task_key)
    missing_columns = required_embedding_columns - set(parquet.schema_arrow.names)
    if missing_columns:
        raise RuntimeError(f"Shard {path} is missing {sorted(missing_columns)}")
    embedding_type = parquet.schema_arrow.field("embedding").type
    if not pa.types.is_list(embedding_type) or embedding_type.value_type != pa.float32():
        raise RuntimeError(f"Shard {path} has embedding type {embedding_type}")
    embedding_files.append(path)

print("Pipeline complete at snapshot:", run_complete_at_snapshot)
print(f"Published {TARGET_WINDOW} task shards: {len(embedding_files)}")

In [ ]:
fields = read_parquet_file(
    OUTPUT_DIR / "fields.parquet",
    columns=[
        "field_uid", "id", "landcover", "wkt", "utm_epsg",
        "pixel_count", "duplicate_count", "geometry_sha256",
    ],
)
pixels = read_parquet_file(
    OUTPUT_DIR / "pixels.parquet",
    columns=[
        "pixel_id", "utm_epsg", "pixel_x_index", "pixel_y_index",
        "pixel_longitude", "pixel_latitude",
    ],
)
memberships = read_parquet_file(
    OUTPUT_DIR / "field_pixels.parquet",
    columns=["field_uid", "pixel_id", "overlap_field_count", "label_conflict"],
)

required_field_columns = {
    "field_uid", "id", "landcover", "wkt", "utm_epsg",
    "pixel_count", "duplicate_count", "geometry_sha256",
}
required_membership_columns = {
    "field_uid", "pixel_id", "overlap_field_count", "label_conflict",
}
required_pixel_columns = {
    "pixel_id", "utm_epsg", "pixel_x_index", "pixel_y_index",
    "pixel_longitude", "pixel_latitude",
}
if missing := required_field_columns - set(fields.columns):
    raise RuntimeError(f"fields.parquet is missing {sorted(missing)}")
if missing := required_membership_columns - set(memberships.columns):
    raise RuntimeError(f"field_pixels.parquet is missing {sorted(missing)}")
if missing := required_pixel_columns - set(pixels.columns):
    raise RuntimeError(f"pixels.parquet is missing {sorted(missing)}")
if fields["field_uid"].duplicated().any():
    raise RuntimeError("fields.parquet contains duplicate field_uid values.")
fields["landcover"] = fields["landcover"].map(lambda value: str(value).strip())
if fields["landcover"].eq("").any():
    raise RuntimeError("fields.parquet contains an empty field label.")
membership_keys = ["field_uid", "pixel_id"]
if memberships.duplicated(membership_keys).any():
    raise RuntimeError("field_pixels.parquet contains duplicate memberships.")
if pixels["pixel_id"].duplicated().any():
    raise RuntimeError("pixels.parquet contains duplicate pixel_id values.")
if not memberships["field_uid"].isin(fields["field_uid"]).all():
    raise RuntimeError("field_pixels.parquet references an unknown field_uid.")
if not memberships["pixel_id"].isin(pixels["pixel_id"]).all():
    raise RuntimeError("field_pixels.parquet references an unknown pixel_id.")

expected_pair_index = pd.MultiIndex.from_frame(memberships[membership_keys])
expected_pixels = memberships.groupby("field_uid", sort=False).size()
declared_pixels = fields.set_index("field_uid")["pixel_count"].astype(int)
if not expected_pixels.sort_index().equals(
    declared_pixels.reindex(expected_pixels.index).sort_index()
):
    raise RuntimeError("Declared field pixel counts disagree with memberships.")
recomputed_static_pixel_ids = (
    "utm-"
    + pixels["utm_epsg"].astype(np.int64).astype(str)
    + "-10m-"
    + pixels["pixel_x_index"].astype(np.int64).astype(str)
    + "-"
    + pixels["pixel_y_index"].astype(np.int64).astype(str)
)
if not pixels["pixel_id"].astype(str).eq(recomputed_static_pixel_ids).all():
    raise RuntimeError("pixels.parquet grid coordinates do not reproduce pixel_id.")
field_lookup = fields.set_index("field_uid")
pixel_lookup = pixels.set_index("pixel_id")

index_columns = [
    "row_id", "run_fingerprint", "field_uid", "pixel_id",
    "utm_epsg", "pixel_x_index", "pixel_y_index", "window_id",
    "pixel_longitude", "pixel_latitude", "landcover", "outcome",
]
seen_memberships = np.zeros(len(memberships), dtype=bool)
index_parts = []
for path_code, path in enumerate(embedding_files):
    part = read_parquet_file(path, columns=index_columns)
    if part.empty:
        raise RuntimeError(f"Published shard is empty: {path}")
    if not part["run_fingerprint"].eq(run_fingerprint).all():
        raise RuntimeError(f"Shard row run fingerprint mismatch: {path}")
    if not part["window_id"].eq(TARGET_WINDOW).all():
        raise RuntimeError(f"Shard row window mismatch: {path}")
    if not set(part["outcome"].unique()) <= {"complete", "empty_window"}:
        raise RuntimeError(f"Unknown shard outcome in {path}")
    part["landcover"] = part["landcover"].map(lambda value: str(value).strip())
    expected_labels = part["field_uid"].map(field_lookup["landcover"])
    if expected_labels.isna().any() or not part["landcover"].eq(expected_labels).all():
        raise RuntimeError(f"Shard labels disagree with fields.parquet: {path}")
    expected_grid = pixel_lookup.reindex(part["pixel_id"].astype(str))
    if expected_grid["utm_epsg"].isna().any():
        raise RuntimeError(f"Shard references an unknown pixels.parquet pixel: {path}")
    for column in ("utm_epsg", "pixel_x_index", "pixel_y_index"):
        if not np.array_equal(
            part[column].to_numpy(dtype=np.int64),
            expected_grid[column].to_numpy(dtype=np.int64),
        ):
            raise RuntimeError(f"Shard {column} disagrees with pixels.parquet: {path}")
    for column in ("pixel_longitude", "pixel_latitude"):
        if not np.allclose(
            part[column].to_numpy(dtype=np.float64),
            expected_grid[column].to_numpy(dtype=np.float64),
            rtol=0.0,
            atol=1e-10,
        ):
            raise RuntimeError(f"Shard {column} disagrees with pixels.parquet: {path}")
    positions = expected_pair_index.get_indexer(
        pd.MultiIndex.from_frame(part[membership_keys])
    )
    if (positions < 0).any():
        raise RuntimeError(f"Shard contains a row outside field_pixels.parquet: {path}")
    if (
        part["row_id"].duplicated().any()
        or len(np.unique(positions)) != len(positions)
        or seen_memberships[positions].any()
    ):
        raise RuntimeError(f"Duplicate field/pixel/window row in {path}")
    calculated_ids = [
        expected_row_id(run_fingerprint, field_uid, pixel_id, TARGET_WINDOW)
        for field_uid, pixel_id in part[membership_keys].itertuples(index=False, name=None)
    ]
    if not part["row_id"].astype(str).eq(calculated_ids).all():
        raise RuntimeError(f"Deterministic row_id mismatch in {path}")
    seen_memberships[positions] = True
    compact = part[["field_uid", "pixel_id", "outcome"]].copy()
    compact["_path_code"] = np.int32(path_code)
    compact["overlap_field_count"] = memberships.iloc[positions][
        "overlap_field_count"
    ].to_numpy(dtype=np.int32)
    compact["label_conflict"] = memberships.iloc[positions][
        "label_conflict"
    ].to_numpy(dtype=bool)
    index_parts.append(compact)

del part, compact, expected_grid, expected_labels, calculated_ids
pixel_index = pd.concat(index_parts, ignore_index=True)
index_parts.clear()
del index_parts
actual_pixels = pixel_index.groupby("field_uid", sort=False).size()
published_field_ids = expected_pixels.index[
    actual_pixels.reindex(expected_pixels.index, fill_value=0).eq(expected_pixels)
]
published_set = set(published_field_ids)

pixel_index["landcover"] = pixel_index["field_uid"].map(
    field_lookup["landcover"]
)
pixel_index["pixel_count"] = pixel_index["field_uid"].map(
    field_lookup["pixel_count"]
).astype(np.int64)
if pixel_index["landcover"].isna().any():
    raise RuntimeError("A published row references an unknown field_uid.")
overlap_field_ids = set(
    memberships.loc[memberships["overlap_field_count"].gt(1), "field_uid"]
)
geometry_label_counts = fields.groupby("geometry_sha256")["landcover"].nunique()
conflicting_geometry_hashes = set(geometry_label_counts[geometry_label_counts.gt(1)].index)
conflicting_geometry_field_ids = set(
    fields.loc[
        fields["geometry_sha256"].isin(conflicting_geometry_hashes), "field_uid"
    ]
)
conflicting_geometry_pixel_ids = set(
    memberships.loc[
        memberships["field_uid"].isin(conflicting_geometry_field_ids), "pixel_id"
    ]
)
unanimous_geometry_fields = fields[
    ~fields["geometry_sha256"].isin(conflicting_geometry_hashes)
].copy()
unanimous_geometry_fields["_published_priority"] = (
    unanimous_geometry_fields["field_uid"].isin(published_set)
)
unanimous_geometry_fields = unanimous_geometry_fields.sort_values(
    ["geometry_sha256", "_published_priority", "field_uid"],
    ascending=[True, False, True],
    kind="stable",
)
canonical_geometry_field_ids = set(
    unanimous_geometry_fields.groupby("geometry_sha256", sort=False).head(1)[
        "field_uid"
    ]
)
duplicate_geometry_replica_ids = (
    set(unanimous_geometry_fields["field_uid"]) - canonical_geometry_field_ids
)
excluded_field_ids = conflicting_geometry_field_ids | duplicate_geometry_replica_ids
effective_memberships = memberships[
    ~memberships["field_uid"].isin(excluded_field_ids)
].copy()
effective_memberships["landcover"] = effective_memberships["field_uid"].map(
    field_lookup["landcover"]
)
effective_overlap_by_pixel = effective_memberships.groupby("pixel_id")[
    "field_uid"
].nunique()
effective_label_count_by_pixel = effective_memberships.groupby("pixel_id")[
    "landcover"
].nunique()
pixel_index["_fully_published"] = pixel_index["field_uid"].isin(published_set)
pixel_index["_field_clean_for_test"] = ~pixel_index["field_uid"].isin(
    excluded_field_ids
)
pixel_index["_effective_overlap_field_count"] = pixel_index["pixel_id"].map(
    effective_overlap_by_pixel
).fillna(0).astype(np.int32)
pixel_index["_effective_label_conflict"] = pixel_index["pixel_id"].map(
    effective_label_count_by_pixel
).fillna(0).gt(1)
pixel_index["_private_for_test"] = (
    pixel_index["_fully_published"]
    & pixel_index["outcome"].eq("complete")
    & pixel_index["_field_clean_for_test"]
    & ~pixel_index["pixel_id"].isin(conflicting_geometry_pixel_ids)
    & pixel_index["_effective_overlap_field_count"].eq(1)
    & ~pixel_index["_effective_label_conflict"]
)
private_index = pixel_index[pixel_index["_private_for_test"]].copy()
field_stats = (
    private_index.groupby(["field_uid", "landcover"], as_index=False)
    .agg(private_complete_pixels=("pixel_id", "nunique"))
)
field_stats = field_stats.merge(
    fields[["field_uid", "pixel_count"]],
    on="field_uid",
    how="left",
    validate="one_to_one",
)
field_stats = field_stats[
    field_stats["private_complete_pixels"].ge(MIN_PRIVATE_PIXELS_PER_FIELD)
]
crop_counts = (
    field_stats.groupby("landcover")["field_uid"].nunique()
    .sort_values(ascending=False, kind="stable")
)
eligible_crop_counts = crop_counts[crop_counts.ge(MIN_FIELDS_PER_CROP)]
if TARGET_LABELS is not None:
    requested_labels = [str(label).strip() for label in TARGET_LABELS]
    missing_labels = sorted(set(requested_labels) - set(eligible_crop_counts.index))
    if missing_labels:
        raise RuntimeError(
            f"Requested labels lack enough eligible completed fields: {missing_labels}"
        )
    eligible_crop_counts = eligible_crop_counts.reindex(requested_labels)
selected_crops = eligible_crop_counts.head(MAX_CROPS).index.astype(str).tolist()
if len(selected_crops) < 2:
    raise RuntimeError(
        f"Only {len(selected_crops)} crop labels have at least "
        f"{MIN_FIELDS_PER_CROP} fully published fields with "
        f"{MIN_PRIVATE_PIXELS_PER_FIELD}+ private complete pixels. "
        "Wait for more task shards and rerun."
    )

selection_rng = np.random.default_rng(RANDOM_SEED)
selected_field_ids = []
for crop in selected_crops:
    candidates = np.array(
        sorted(field_stats.loc[field_stats["landcover"].eq(crop), "field_uid"]),
        dtype=object,
    )
    if len(candidates) > MAX_FIELDS_PER_CROP:
        candidates = np.sort(
            selection_rng.choice(candidates, size=MAX_FIELDS_PER_CROP, replace=False)
        )
    selected_field_ids.extend(candidates.tolist())
selected_field_set = set(selected_field_ids)

analysis_parts = []
for field_uid, group in private_index[
    private_index["field_uid"].isin(selected_field_set)
].groupby("field_uid", sort=True):
    ordered = group.sort_values("pixel_id", kind="stable")
    if len(ordered) > MAX_PIXELS_PER_FIELD:
        positions = np.linspace(
            0, len(ordered) - 1, MAX_PIXELS_PER_FIELD, dtype=np.int64
        )
        ordered = ordered.iloc[positions]
    analysis_parts.append(ordered)
analysis_index = pd.concat(analysis_parts, ignore_index=True)

visual_field_stats = (
    pixel_index[
        pixel_index["_fully_published"]
        & pixel_index["outcome"].eq("complete")
        & pixel_index["landcover"].isin(selected_crops)
    ]
    .groupby(["field_uid", "landcover"], as_index=False)
    .agg(
        embedded_pixels=("pixel_id", "nunique"),
        shared_pixels=("overlap_field_count", lambda values: int((values > 1).sum())),
        pixel_count=("pixel_count", "first"),
    )
)
example_field_ids = []
for crop in selected_crops:
    candidates = visual_field_stats[
        visual_field_stats["landcover"].eq(crop)
    ].sort_values(
        ["pixel_count", "shared_pixels", "field_uid"],
        ascending=[False, False, True],
        kind="stable",
    )
    if not candidates.empty:
        bounded = candidates[
            candidates["pixel_count"].le(MAX_EXAMPLE_FIELD_PIXELS)
        ]
        if not bounded.empty:
            example_field_ids.append(bounded.iloc[0]["field_uid"])
    if len(example_field_ids) == N_EXAMPLE_FIELDS:
        break
if not example_field_ids:
    raise RuntimeError(
        f"No full example field has <= {MAX_EXAMPLE_FIELD_PIXELS} pixels. "
        "Increase MAX_EXAMPLE_FIELD_PIXELS deliberately if the VM has enough memory."
    )
example_pixel_reference = pixels.rename(
    columns={
        column: f"{column}_expected"
        for column in required_pixel_columns
        if column != "pixel_id"
    }
)
example_index = pixel_index[
    pixel_index["field_uid"].isin(example_field_ids)
].merge(
    example_pixel_reference,
    on="pixel_id",
    how="left",
    validate="many_to_one",
)

analysis_index["row_id"] = [
    expected_row_id(run_fingerprint, field_uid, pixel_id, TARGET_WINDOW)
    for field_uid, pixel_id in analysis_index[membership_keys].itertuples(
        index=False, name=None
    )
]
example_index["row_id"] = [
    expected_row_id(run_fingerprint, field_uid, pixel_id, TARGET_WINDOW)
    for field_uid, pixel_id in example_index[membership_keys].itertuples(
        index=False, name=None
    )
]
analysis_ids = set(analysis_index["row_id"].astype(str))
example_ids = set(example_index["row_id"].astype(str))
requested_index = pd.concat([example_index, analysis_index], ignore_index=True)
requested_index = requested_index.drop_duplicates("row_id")
requested_index["_analysis"] = requested_index["row_id"].isin(analysis_ids)
requested_index["_example"] = requested_index["row_id"].isin(example_ids)

snapshot_summary = pd.DataFrame(
    {
        "value": [
            len(fields),
            len(published_field_ids),
            len(overlap_field_ids),
            len(conflicting_geometry_field_ids & published_set),
            len(duplicate_geometry_replica_ids & published_set),
            len(field_stats),
            len(selected_field_ids),
            len(analysis_index),
        ]
    },
    index=[
        "all fields",
        f"fully published fields in {TARGET_WINDOW}",
        "fields touching shared pixels (only shared rows are excluded)",
        "fields excluded for conflicting labels on identical geometry",
        "same-label geometry replicas excluded (one canonical field retained)",
        "eligible clean fields across all crops",
        "selected fields",
        "selected individual pixel samples",
    ],
)
display(snapshot_summary)
display(
    eligible_crop_counts.rename("eligible_completed_fields").to_frame().head(MAX_CROPS)
)

In [ ]:
# Load the heavy embedding column only for selected analysis/example rows.
embedding_parts = []
for path_code, requested in requested_index.groupby("_path_code", sort=False):
    requested_ids = set(requested["row_id"].astype(str))
    part = pq.read_table(
        embedding_files[int(path_code)],
        columns=["row_id", "embedding"],
        filters=[("row_id", "in", sorted(requested_ids))],
        partitioning=None,
    ).to_pandas()
    embedding_parts.append(part)
embedding_rows = pd.concat(embedding_parts, ignore_index=True)
if embedding_rows["row_id"].duplicated().any():
    raise RuntimeError("Filtered embedding rows contain duplicate row_id values.")
loaded = requested_index.merge(
    embedding_rows, on="row_id", how="left", validate="one_to_one", indicator=True
)
if not loaded["_merge"].eq("both").all():
    raise RuntimeError("Not every selected row was returned by the snapshot read.")
loaded = loaded.drop(columns="_merge")
loaded["_vector"] = loaded["embedding"].map(embedding_vector)
complete_mask = loaded["outcome"].eq("complete")
if loaded.loc[complete_mask, "_vector"].isna().any():
    raise RuntimeError("A complete row lacks a finite float32[128] embedding.")
if loaded.loc[~complete_mask, "_vector"].notna().any():
    raise RuntimeError("An empty_window row unexpectedly contains an embedding.")

analysis = loaded[loaded["_analysis"]].copy().reset_index(drop=True)
if not analysis["outcome"].eq("complete").all():
    raise RuntimeError("The crop-separation sample contains a non-complete pixel.")
if analysis["pixel_id"].duplicated().any():
    raise RuntimeError(
        "A physical pixel appears in more than one analysis field; leakage guard failed."
    )
raw_features = np.stack(analysis["_vector"].to_numpy()).astype(np.float64)
feature_norms = np.linalg.norm(raw_features, axis=1, keepdims=True)
if (feature_norms <= 0).any():
    raise RuntimeError("A selected embedding has zero norm.")
features = raw_features / feature_norms
analysis["_feature"] = list(features)

pca_mean = features.mean(axis=0, keepdims=True)
pca_centered = features - pca_mean
_, singular_values, right_vectors = np.linalg.svd(pca_centered, full_matrices=False)
rank_tolerance = (
    np.finfo(features.dtype).eps * max(pca_centered.shape) * singular_values[0]
)
supported_components = int(np.count_nonzero(singular_values > rank_tolerance))
if supported_components < 2:
    raise RuntimeError("Selected pixel embeddings do not support a 2-D PCA view.")
pca_component_count = min(3, supported_components)
pca_components = right_vectors[:pca_component_count]
pca_scores = pca_centered @ pca_components.T
analysis["pca_1"] = pca_scores[:, 0]
analysis["pca_2"] = pca_scores[:, 1]
pca_low = np.quantile(pca_scores, 0.02, axis=0)
pca_high = np.quantile(pca_scores, 0.98, axis=0)
pca_spread = np.where(pca_high > pca_low, pca_high - pca_low, 1.0)
scaled_scores = np.clip((pca_scores - pca_low) / pca_spread, 0.0, 1.0)
analysis_rgb = np.full((len(analysis), 3), 0.5, dtype=np.float32)
analysis_rgb[:, :pca_component_count] = scaled_scores
analysis["_rgb"] = list(analysis_rgb)

example_rows = loaded[loaded["_example"]].copy().reset_index(drop=True)
example_rows["pca_1"] = np.nan
example_rows["pca_2"] = np.nan
example_rows["_rgb"] = None
example_complete = example_rows["outcome"].eq("complete").to_numpy()
if example_complete.any():
    example_raw = np.stack(example_rows.loc[example_complete, "_vector"].to_numpy()).astype(np.float64)
    example_norms = np.linalg.norm(example_raw, axis=1, keepdims=True)
    if (example_norms <= 0).any():
        raise RuntimeError("A complete example embedding has zero norm.")
    example_features = example_raw / example_norms
    example_scores = (example_features - pca_mean) @ pca_components.T
    example_rgb = np.full((len(example_scores), 3), 0.5, dtype=np.float32)
    example_scaled = np.clip(
        (example_scores - pca_low) / pca_spread, 0.0, 1.0
    )
    example_rgb[:, :pca_component_count] = example_scaled
    example_rows.loc[example_complete, "pca_1"] = example_scores[:, 0]
    example_rows.loc[example_complete, "pca_2"] = example_scores[:, 1]
    example_rows.loc[example_complete, "_rgb"] = pd.Series(
        list(example_rgb), index=example_rows.index[example_complete], dtype=object
    )

grid_columns = [
    "pixel_id", "utm_epsg_expected",
    "pixel_x_index_expected", "pixel_y_index_expected",
]
pixel_grids = example_rows[grid_columns].drop_duplicates().rename(
    columns={
        "utm_epsg_expected": "utm_epsg",
        "pixel_x_index_expected": "pixel_x_index",
        "pixel_y_index_expected": "pixel_y_index",
    }
)
if pixel_grids["pixel_id"].duplicated().any():
    raise RuntimeError("An example pixel has conflicting grid metadata.")
footprint_by_pixel = {}
for epsg, group in pixel_grids.groupby("utm_epsg", sort=True):
    left = group["pixel_x_index"].to_numpy(dtype=np.float64) * PIXEL_SIZE_M
    bottom = group["pixel_y_index"].to_numpy(dtype=np.float64) * PIXEL_SIZE_M
    right = left + PIXEL_SIZE_M
    top = bottom + PIXEL_SIZE_M
    corner_x = np.column_stack([left, right, right, left])
    corner_y = np.column_stack([bottom, bottom, top, top])
    transformer = Transformer.from_crs(int(epsg), 4326, always_xy=True)
    longitude, latitude = transformer.transform(corner_x.ravel(), corner_y.ravel())
    footprints = np.column_stack([longitude, latitude]).reshape(-1, 4, 2)
    footprint_by_pixel.update(
        zip(group["pixel_id"].astype(str), footprints, strict=True)
    )
example_rows["_footprint"] = example_rows["pixel_id"].astype(str).map(
    footprint_by_pixel
)
if example_rows["_footprint"].map(lambda value: not isinstance(value, np.ndarray)).any():
    raise RuntimeError("An example pixel footprint could not be reconstructed.")

analysis_summary = (
    analysis.groupby("landcover")
    .agg(fields=("field_uid", "nunique"), pixel_samples=("pixel_id", "nunique"))
    .reindex(selected_crops)
)
display(analysis_summary)
print(
    f"PCA uses {len(analysis):,} individual pixel vectors from "
    f"{analysis['field_uid'].nunique():,} completed fields. No field averaging."
)

In [ ]:
crop_palette = plt.get_cmap("tab10")
crop_colors = {
    crop: crop_palette(index % 10) for index, crop in enumerate(selected_crops)
}

figure, axis = plt.subplots(figsize=(11, 8))
for crop in selected_crops:
    rows = analysis[analysis["landcover"].eq(crop)]
    axis.scatter(
        rows["pca_1"],
        rows["pca_2"],
        s=12,
        alpha=0.55,
        linewidths=0,
        color=crop_colors[crop],
        label=f"{crop} ({len(rows):,} pixels / {rows['field_uid'].nunique()} fields)",
    )
axis.set_title(
    f"{TARGET_WINDOW}: individual 10 m pixel embeddings (PCA view)\n"
    "Every dot is one pixel; color is the parent field's crop label"
)
axis.set_xlabel("pixel embedding PC1")
axis.set_ylabel("pixel embedding PC2")
axis.grid(alpha=0.15)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
figure.tight_layout()
plt.show()

In [ ]:
def polygon_parts(geometry):
    if geometry.geom_type == "Polygon":
        yield geometry
    elif hasattr(geometry, "geoms"):
        for child in geometry.geoms:
            yield from polygon_parts(child)

def draw_field_outline(axis, geometry):
    for polygon in polygon_parts(geometry):
        x, y = polygon.exterior.xy
        axis.plot(x, y, color="black", linewidth=1.3, zorder=5)
        for interior in polygon.interiors:
            x, y = interior.xy
            axis.plot(x, y, color="black", linewidth=0.8, zorder=5)

def draw_footprints(axis, rows, facecolors, edgecolors="white", zorder=2):
    if rows.empty:
        return
    axis.add_collection(
        PolyCollection(
            np.stack(rows["_footprint"].to_numpy()),
            facecolors=facecolors,
            edgecolors=edgecolors,
            linewidths=0.25,
            zorder=zorder,
        )
    )

figure, axes = plt.subplots(
    len(example_field_ids), 2, figsize=(13, 4.3 * len(example_field_ids)), squeeze=False
)
for row_number, field_uid in enumerate(example_field_ids):
    rows = example_rows[example_rows["field_uid"].eq(field_uid)].copy()
    metadata = fields.set_index("field_uid").loc[field_uid]
    geometry, _ = parse_field_geometry(str(metadata["wkt"]))
    map_axis, embedding_axis = axes[row_number]

    complete = rows[rows["outcome"].eq("complete")]
    empty = rows[rows["outcome"].eq("empty_window")]
    if not complete.empty:
        draw_footprints(map_axis, complete, np.stack(complete["_rgb"].to_numpy()))
    if not empty.empty:
        draw_footprints(map_axis, empty, "#bdbdbd", zorder=1)
    shared = rows[rows["overlap_field_count"].gt(1)]
    if not shared.empty:
        draw_footprints(
            map_axis, shared, "none", edgecolors="#d62728", zorder=4
        )
    draw_field_outline(map_axis, geometry)

    footprints = np.stack(rows["_footprint"].to_numpy())
    min_x = min(float(footprints[:, :, 0].min()), geometry.bounds[0])
    min_y = min(float(footprints[:, :, 1].min()), geometry.bounds[1])
    max_x = max(float(footprints[:, :, 0].max()), geometry.bounds[2])
    max_y = max(float(footprints[:, :, 1].max()), geometry.bounds[3])
    pad_x = max((max_x - min_x) * 0.06, 1e-6)
    pad_y = max((max_y - min_y) * 0.06, 1e-6)
    map_axis.set_xlim(min_x - pad_x, max_x + pad_x)
    map_axis.set_ylim(min_y - pad_y, max_y + pad_y)
    mean_latitude = (min_y + max_y) / 2
    map_axis.set_aspect(
        1.0 / max(abs(np.cos(np.deg2rad(mean_latitude))), 1e-6),
        adjustable="box",
    )
    map_axis.set_title(
        f"Field id={metadata['id']} — {metadata['landcover']}\n"
        f"{len(rows)} actual 10 m cells: {len(complete)} embedded, "
        f"{len(empty)} empty, {len(shared)} shared (red edge)"
    )
    map_axis.set_xlabel("longitude")
    map_axis.set_ylabel("latitude")

    embedding_axis.scatter(
        analysis["pca_1"], analysis["pca_2"], s=5, color="0.85", alpha=0.25
    )
    if not complete.empty:
        embedding_axis.scatter(
            complete["pca_1"],
            complete["pca_2"],
            s=45,
            c=np.stack(complete["_rgb"].to_numpy()),
            edgecolors="black",
            linewidths=0.45,
            zorder=3,
        )
    embedding_axis.set_title(
        f"The same {len(complete)} embedded field pixels in PCA space"
    )
    embedding_axis.set_xlabel("pixel embedding PC1")
    embedding_axis.set_ylabel("pixel embedding PC2")
    embedding_axis.grid(alpha=0.15)

figure.suptitle(
    f"{TARGET_WINDOW}: real field pixel footprints ↔ the same individual embeddings",
    fontsize=15,
    y=1.002,
)
figure.tight_layout()
plt.show()

In [ ]:
# Deterministic crop-stratified folds are assigned to fields, then inherited by pixels.
field_table = analysis[["field_uid", "landcover"]].drop_duplicates().copy()
if field_table["field_uid"].duplicated().any():
    raise RuntimeError("A field has more than one crop label.")
if field_table.groupby("landcover").size().min() < N_FOLDS:
    raise RuntimeError(f"Every selected crop needs at least {N_FOLDS} fields.")
fold_rng = np.random.default_rng(RANDOM_SEED + 1)
field_to_fold = {}
for crop in selected_crops:
    field_ids = np.array(
        sorted(field_table.loc[field_table["landcover"].eq(crop), "field_uid"]),
        dtype=object,
    )
    fold_rng.shuffle(field_ids)
    for position, field_uid in enumerate(field_ids):
        field_to_fold[field_uid] = position % N_FOLDS
field_table["fold"] = field_table["field_uid"].map(field_to_fold).astype(int)
analysis["fold"] = analysis["field_uid"].map(field_to_fold).astype(int)
classes = np.array(selected_crops, dtype=object)
model_features = np.stack(analysis["_feature"].to_numpy())
field_ids = analysis["field_uid"].astype(str).to_numpy()
pixel_ids = analysis["pixel_id"].astype(str).to_numpy()
row_ids = analysis["row_id"].astype(str).to_numpy()
folds = analysis["fold"].to_numpy()
true_labels = analysis["landcover"].astype(str).to_numpy()

def grouped_nearest_centroid_predictions(features, labels):
    predictions = np.empty(len(labels), dtype=object)
    all_distances = np.empty((len(labels), len(classes)), dtype=np.float64)
    for fold in range(N_FOLDS):
        test_mask = folds == fold
        train_mask = ~test_mask
        if set(field_ids[train_mask]) & set(field_ids[test_mask]):
            raise RuntimeError("field_uid leakage between train and test.")
        if set(pixel_ids[train_mask]) & set(pixel_ids[test_mask]):
            raise RuntimeError("physical pixel leakage between train and test.")
        if set(row_ids[train_mask]) & set(row_ids[test_mask]):
            raise RuntimeError("row_id leakage between train and test.")

        train_x = features[train_mask]
        test_x = features[test_mask]
        train_labels = labels[train_mask]
        train_fields = field_ids[train_mask]

        field_pixel_counts = pd.Series(train_fields).value_counts().to_dict()
        field_labels = pd.DataFrame(
            {"field_uid": train_fields, "label": train_labels}
        ).drop_duplicates()
        fields_per_class = field_labels["label"].value_counts().to_dict()
        train_weights = np.array(
            [
                1.0 / (field_pixel_counts[field_uid] * fields_per_class[label])
                for field_uid, label in zip(train_fields, train_labels, strict=True)
            ]
        )
        train_mean = np.average(train_x, axis=0, weights=train_weights)
        train_variance = np.average(
            (train_x - train_mean) ** 2, axis=0, weights=train_weights
        )
        train_scale = np.sqrt(train_variance)
        train_scale[train_scale < 1e-8] = 1.0
        train_z = (train_x - train_mean) / train_scale
        test_z = (test_x - train_mean) / train_scale
        centroids = []
        for crop in classes:
            class_mask = train_labels == crop
            if not class_mask.any():
                raise RuntimeError(f"Training fold lacks crop class {crop}.")
            centroids.append(
                np.average(
                    train_z[class_mask],
                    axis=0,
                    weights=train_weights[class_mask],
                )
            )
        centroids = np.stack(centroids)
        distances = (
            (test_z * test_z).sum(axis=1, keepdims=True)
            + (centroids * centroids).sum(axis=1)[None, :]
            - 2.0 * test_z @ centroids.T
        )
        predictions[test_mask] = classes[np.argmin(distances, axis=1)]
        all_distances[test_mask] = distances
    return predictions, all_distances

def classification_metrics(actual, predicted, weights):
    confusion = np.zeros((len(classes), len(classes)), dtype=np.float64)
    class_index = {label: index for index, label in enumerate(classes)}
    for actual_label, predicted_label, weight in zip(
        actual, predicted, weights, strict=True
    ):
        confusion[class_index[actual_label], class_index[predicted_label]] += weight
    row_totals = confusion.sum(axis=1)
    column_totals = confusion.sum(axis=0)
    true_positive = np.diag(confusion)
    recall = np.divide(
        true_positive, row_totals, out=np.zeros_like(true_positive), where=row_totals > 0
    )
    precision = np.divide(
        true_positive,
        column_totals,
        out=np.zeros_like(true_positive),
        where=column_totals > 0,
    )
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(true_positive),
        where=(precision + recall) > 0,
    )
    return {
        "balanced_accuracy": float(recall.mean()),
        "macro_f1": float(f1.mean()),
        "recall": recall,
        "confusion": confusion,
    }

def field_balanced_pixel_metrics(actual, predicted):
    pixel_counts_by_field = pd.Series(field_ids).value_counts().to_dict()
    weights = np.array(
        [1.0 / pixel_counts_by_field[field_uid] for field_uid in field_ids]
    )
    return classification_metrics(actual, predicted, weights)

oof_predictions, oof_distances = grouped_nearest_centroid_predictions(
    model_features, true_labels
)
observed = field_balanced_pixel_metrics(true_labels, oof_predictions)

field_prediction_rows = []
for field_uid, positions in pd.Series(np.arange(len(analysis))).groupby(analysis["field_uid"]):
    positions = positions.to_numpy()
    mean_distances = oof_distances[positions].mean(axis=0)
    field_prediction_rows.append(
        {
            "field_uid": field_uid,
            "actual": true_labels[positions[0]],
            "predicted": classes[np.argmin(mean_distances)],
        }
    )
field_predictions = pd.DataFrame(field_prediction_rows)
field_observed = classification_metrics(
    field_predictions["actual"].to_numpy(),
    field_predictions["predicted"].to_numpy(),
    np.ones(len(field_predictions), dtype=np.float64),
)

permutation_rng = np.random.default_rng(RANDOM_SEED + 2)
permutation_scores = []
for _ in range(N_PERMUTATIONS):
    permuted_field_labels = {}
    for fold, group in field_table.groupby("fold", sort=True):
        shuffled_labels = group["landcover"].astype(str).to_numpy().copy()
        permutation_rng.shuffle(shuffled_labels)
        permuted_field_labels.update(
            zip(group["field_uid"].astype(str), shuffled_labels, strict=True)
        )
    permuted_labels = np.array(
        [permuted_field_labels[field_uid] for field_uid in field_ids], dtype=object
    )
    permuted_predictions, _ = grouped_nearest_centroid_predictions(
        model_features, permuted_labels
    )
    permutation_scores.append(
        field_balanced_pixel_metrics(permuted_labels, permuted_predictions)["balanced_accuracy"]
    )
permutation_scores = np.asarray(permutation_scores)
permutation_p_value = (
    1 + np.count_nonzero(permutation_scores >= observed["balanced_accuracy"])
) / (N_PERMUTATIONS + 1)

display(
    pd.Series(
        {
            "field-balanced pixel balanced accuracy": observed["balanced_accuracy"],
            "field-balanced pixel macro F1": observed["macro_f1"],
            "field mean-distance balanced accuracy": field_observed["balanced_accuracy"],
            "field mean-distance macro F1": field_observed["macro_f1"],
            f"field-label permutation p-value ({N_PERMUTATIONS} permutations)": permutation_p_value,
        },
        name="score",
    ).to_frame()
)

In [ ]:
confusion = observed["confusion"]
row_normalized = np.divide(
    confusion,
    confusion.sum(axis=1, keepdims=True),
    out=np.zeros_like(confusion),
    where=confusion.sum(axis=1, keepdims=True) > 0,
)
figure, axes = plt.subplots(1, 3, figsize=(20, 5.5))
image = axes[0].imshow(row_normalized, vmin=0, vmax=1, cmap="Blues")
axes[0].set_xticks(range(len(classes)), classes, rotation=45, ha="right")
axes[0].set_yticks(range(len(classes)), classes)
axes[0].set_xlabel("predicted crop")
axes[0].set_ylabel("true crop")
axes[0].set_title("OOF field-balanced pixel confusion")
for row in range(len(classes)):
    for column in range(len(classes)):
        value = row_normalized[row, column]
        axes[0].text(
            column, row, f"{value:.2f}", ha="center", va="center",
            color="white" if value > 0.5 else "black", fontsize=8,
        )
figure.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].barh(classes, observed["recall"], color=[crop_colors[crop] for crop in classes])
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("recall")
axes[1].set_title("Per-crop held-out pixel recall")
axes[1].grid(axis="x", alpha=0.2)

axes[2].hist(permutation_scores, bins=10, color="0.65", edgecolor="white")
axes[2].axvline(
    observed["balanced_accuracy"], color="#d62728", linewidth=2,
    label=f"observed={observed['balanced_accuracy']:.3f}",
)
axes[2].set_xlabel("balanced accuracy")
axes[2].set_ylabel("permutations")
axes[2].set_title(f"Field-label permutation null (p={permutation_p_value:.3f})")
axes[2].legend()
figure.suptitle(
    "Crop separation from individual pixel embeddings — field-grouped held-out folds",
    fontsize=14,
)
figure.tight_layout()
plt.show()

display(
    pd.DataFrame(
        {
            "crop": classes,
            "held_out_recall": observed["recall"],
            "selected_fields": [
                field_table["landcover"].eq(crop).sum() for crop in classes
            ],
            "selected_pixels": [analysis["landcover"].eq(crop).sum() for crop in classes],
        }
    ).set_index("crop")
)

## What the figures mean

- Every colored map polygon is an actual 10 × 10 m UTM grid cell transformed to WGS84. The black line is the field WKT. Cell edges can cross the WKT because membership uses the cell center.
- Every PCA marker is one pixel embedding. No field embedding is created and no field's pixel vectors are averaged for the plots or model input.
- The left/right example panels are linked: the highlighted points on the right are the same cells drawn on the left. PCA-RGB is only an embedding-similarity diagnostic, not satellite true color.
- Red map edges identify pixels shared with another field. They remain visible in the footprint plot but are excluded from crop testing. Gray cells are published `empty_window` rows.
- Cross-validation holds out entire `field_uid` values. Shared pixel rows are removed while private rows from those fields are retained. Same-label geometry duplicates collapse to one canonical field; conflicting-label duplicate fields and every pixel they touch are removed. Train/test field, pixel, and row IDs are asserted disjoint.
- The nearest-crop-centroid baseline uses the full normalized 128-D pixel vectors. Weighted standardization is fitted only on training fields. Training weights give each field equal influence within its crop; reported pixel metrics give each held-out field equal total weight.
- The field score selects the crop with the lowest mean held-out per-pixel class distance; it does not average embeddings. The permutation test shuffles labels at field level within folds. Its p-value is a conditional Monte Carlo diagnostic for this selected partial snapshot.
- This is an exploratory within-snapshot separation check, not a spatial-generalization estimate. Folds separate fields but do not block neighboring fields by geography, so site/acquisition context can contribute to the score and the permutation p-value is not spatially exchangeable. Rerun after more shards finish and use fixed coarse spatial blocks plus an independent geographic/temporal test set before making a scientific claim.